# Exploration du jeu de données : *Escherichia coli* - Ciprofloxacine

**Auteur :** Benjamin HIN  
**Projet :** Machine Learning / Deep Learning pour la prédiction de la résistance aux antibiotiques (AMR)  
**Jeu de données :** 

---

### Objectifs du notebook :
1. **Importer** le jeu de données pré-traité .
2. **Afficher** les données et inspecter leur organisation.
3. **Fournir un résumé complet** : dimensions, métadonnées, valeurs manquantes, types de données.
4. **Analyser la variable cible ()** : distribution de la résistance vs sensibilité à la ciprofloxacine.
5. **Explorer les métadonnées génomiques & MLST** (, , ).
6. **Analyser les variables génomiques** (matrice binaire de présence/absence de gènes et clusters).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuration de l'affichage
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid", font_scale=1.1)

print("Bibliothèques chargées avec succès.")


Bibliothèques chargées avec succès.


## 1. Importation du jeu de données

In [2]:
# Définition du chemin relatif (compatible selon le répertoire courant d'exécution)
data_path = Path("../data/processed/ecoli_ciprofloxacin.csv")
if not data_path.exists():
    data_path = Path("data/processed/ecoli_ciprofloxacin.csv")

print(f"Lecture du fichier depuis : {data_path.resolve()}")
df = pd.read_csv(data_path)
print(f"Jeu de données chargé : {df.shape[0]:,} lignes et {df.shape[1]:,} colonnes.")


Lecture du fichier depuis : /Users/hin.benjamin/Projet_ML_DL/Projet_ML_DL/data/processed/ecoli_ciprofloxacin.csv
Jeu de données chargé : 5,935 lignes et 20,735 colonnes.


## 2. Affichage du jeu de données

Aperçu des 5 premières lignes du jeu de données (métadonnées et début des features génomiques) :

In [ ]:
df.head()

## 3. Résumé général et structure du jeu de données

In [ ]:
metadata_cols = ['sra', 'Genome.Name', 'mlst_group', 'y']
feature_cols = [c for c in df.columns if c not in metadata_cols]

print("=" * 60)
print("RÉSUMÉ GLOBAL DU DATASET")
print("=" * 60)
print(f"• Nombre total d'échantillons (lignes) : {df.shape[0]:,}")
print(f"• Nombre total de colonnes            : {df.shape[1]:,}")
print(f"• Colonnes de métadonnées / cible     : {len(metadata_cols)} ({', '.join(metadata_cols)})")
print(f"• Caractéristiques génomiques         : {len(feature_cols):,} features")
print(f"• Valeurs manquantes totales (NaN)    : {df.isna().sum().sum()}")
print(f"• Empreinte mémoire estimée           : {df.memory_usage(deep=True).sum() / (1024**2):.2f} Mo")
print("=" * 60)


In [ ]:
# Aperçu des métadonnées
df[metadata_cols].info()


## 4. Analyse de la variable cible  (Résistance à la ciprofloxacine)

La variable  indique le phénotype de résistance aux antibiotiques :
- **0** : Sensible à la ciprofloxacine
- **1** : Résistant à la ciprofloxacine


In [ ]:
target_counts = df['y'].value_counts()
target_pct = df['y'].value_counts(normalize=True) * 100

summary_y = pd.DataFrame({
    'Phénotype': ['Sensible (0)', 'Résistant (1)'],
    'Effectif': [target_counts.get(0, 0), target_counts.get(1, 0)],
    'Proportion (%)': [target_pct.get(0, 0), target_pct.get(1, 0)]
})
summary_y['Proportion (%)'] = summary_y['Proportion (%)'].round(2)
display(summary_y)

# Visualisation graphique
plt.figure(figsize=(7, 4.5))
palette = ['#2ecc71', '#e74c3c']
ax = sns.barplot(data=summary_y, x='Phénotype', y='Effectif', palette=palette, hue='Phénotype', legend=False)

for p in ax.patches:
    height = p.get_height()
    pct = height / len(df) * 100
    ax.annotate(f"{int(height):,} ({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', fontsize=12, color='white', fontweight='bold')

plt.title("Distribution de la résistance à la ciprofloxacine (y)", fontsize=14, pad=15)
plt.ylabel("Nombre d'isolats")
plt.xlabel("")
plt.tight_layout()
plt.show()


## 5. Analyse des métadonnées (, , )

In [ ]:
print(f"Nombre d'identifiants SRA uniques         : {df['sra'].nunique()}")
print(f"Nombre de noms de génomes uniques        : {df['Genome.Name'].nunique()}")
print(f"Nombre de groupes MLST distincts         : {df['mlst_group'].nunique()}")

# Top 10 des types de séquences MLST les plus fréquents
top_mlst = df['mlst_group'].value_counts().head(10)

plt.figure(figsize=(10, 4.5))
sns.barplot(x=top_mlst.values, y=top_mlst.index, palette="mako", hue=top_mlst.index, legend=False)
plt.title("Top 10 des groupes MLST les plus représentés", fontsize=13)
plt.xlabel("Nombre d'isolats")
plt.ylabel("Groupe MLST")
plt.tight_layout()
plt.show()


## 6. Analyse des caractéristiques génomiques

Les 20 731 colonnes génomiques représentent la présence (1) ou l'absence (0) de gènes et de clusters pan-génomiques.


In [ ]:
# Calcul de la fréquence de présence de chaque feature génomique
gene_sums = df[feature_cols].sum(axis=0)
gene_freq = gene_sums / len(df)

sparsity = (df[feature_cols].values == 0).mean() * 100
print(f"• Sparsité globale de la matrice génomique : {sparsity:.2f}% de zéros")
print(f"• Nombre moyen de gènes/clusters présents par isolat : {df[feature_cols].sum(axis=1).mean():.1f}")
print(f"• Min / Max de gènes par isolat : {df[feature_cols].sum(axis=1).min()} / {df[feature_cols].sum(axis=1).max()}")

# Distribution de la fréquence des gènes dans la population
plt.figure(figsize=(9, 4.5))
sns.histplot(gene_freq, bins=50, color="#3498db", kde=False)
plt.title("Distribution de la fréquence des gènes/clusters dans la population", fontsize=13)
plt.xlabel("Fréquence de présence (0 = absent partout, 1 = présent chez tous)")
plt.ylabel("Nombre de features génomiques")
plt.tight_layout()
plt.show()


In [ ]:
# Calcul des corrélations rapides avec le phénotype y (Top 10 gènes les plus associés)
corr_with_y = df[feature_cols].apply(lambda col: col.corr(df['y']))
top_resistant_genes = corr_with_y.sort_values(ascending=False).head(10)
top_susceptible_genes = corr_with_y.sort_values(ascending=True).head(10)

top_corr_df = pd.DataFrame({
    'Feature': top_resistant_genes.index,
    'Corrélation avec y': top_resistant_genes.values.round(3)
})
print("Top 10 des caractéristiques génomiques les plus corrélées positivement à la résistance (y=1) :")
display(top_corr_df)


## 7. Synthèse et observations clés

1. **Dimensions :** Le jeu de données contient **5 935 isolats bactériens** et **20 735 colonnes** (4 colonnes de métadonnées et 20 731 features génomiques).
2. **Qualité des données :** **0 valeur manquante**, données propres et encodées en binaire (0/1) pour les features génomiques.
3. **Déséquilibre de classe :**
   - **Sensibles (0) :** 4 650 échantillons (~78.3 %)
   - **Résistants (1) :** 1 285 échantillons (~21.7 %)
   - *Implication ML/DL :* Prévoir des métriques adaptées aux classes déséquilibrées (F1-score, PR-AUC, Balanced Accuracy) et éventuellement des techniques de pondération de classe () ou de ré-échantillonnage.
4. **Grande dimensionnalité ( \gg n$) :** Avec plus de 20 000 features pour ~6 000 échantillons, des techniques de sélection de features, régularisation (Lasso/ElasticNet), ou architectures adaptées (MLP régularisé, XGBoost/LightGBM, Random Forest) seront pertinentes.
